# SS3 Part B: Applying the SS2 Model to Real STADIOchoice Data

This notebook applies the final model selected in SS2 (XGBoost) to the real STADIOchoice data extract, as scoped in SS1's data request and now delivered by the client.

## 1. Dataset loading

The client extract (`datasets/raw/stadiochoice_ss3/`) contains six tables, matching the SS1 Part C data request exactly: subscription and billing, streaming viewing logs, cancellation and retention, customer support, content catalogue, and search queries. `household_id` is the join key across tables (except the content catalogue, keyed by `title_id`).

The primary entity is the household, one row per household in `table1_subscription_billing.csv`: **4,485 households**. The target is `is_active` (the client's own README confirms churn is equivalent to `is_active = FALSE`, i.e. `churn_date` populated). Of the 4,485 households, **849 have churned and 3,636 are still active, a churn rate of about 18.9%**. This is a meaningfully lower churn rate than SS2's Telco proxy dataset (26.5%), and closer to STADIOchoice's own reported 9.2% monthly streaming churn figure from the SS1 briefing pack, since this is an extract of the full subscriber base rather than a single month's worth of churn events.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "datasets/raw/stadiochoice_ss3"
EXTRACT_DATE = pd.Timestamp("2026-08-31")  # per the client's own README

t1 = pd.read_csv(f"{DATA_DIR}/table1_subscription_billing.csv")
t2 = pd.read_csv(f"{DATA_DIR}/table2_viewing_logs.csv")
t3 = pd.read_csv(f"{DATA_DIR}/table3_cancellation_retention.csv")
t4 = pd.read_csv(f"{DATA_DIR}/table4_customer_support.csv")
t5 = pd.read_csv(f"{DATA_DIR}/table5_content_catalogue.csv")
t6 = pd.read_csv(f"{DATA_DIR}/table6_search_queries.csv")

for name, df in [("table1 subscription/billing", t1), ("table2 viewing logs", t2),
                 ("table3 cancellation/retention", t3), ("table4 customer support", t4),
                 ("table5 content catalogue", t5), ("table6 search queries", t6)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

churn_counts = t1["is_active"].value_counts()
print(f"\nTarget distribution (is_active): {churn_counts.to_dict()}")
print(f"Churn rate: {(~t1['is_active']).mean():.1%}")

table1 subscription/billing: 4485 rows, 12 columns
table2 viewing logs: 44525 rows, 7 columns
table3 cancellation/retention: 988 rows, 7 columns
table4 customer support: 3149 rows, 6 columns
table5 content catalogue: 640 rows, 6 columns
table6 search queries: 13587 rows, 4 columns

Target distribution (is_active): {True: 3636, False: 849}
Churn rate: 18.9%


## 2. Dataset preprocessing

This extract is explicitly flagged by the client as uncleaned operational data, and that holds up under inspection. Each cleaning step below is driven by something actually found in the data, not applied speculatively.

### 2.1 Duplicate rows

`table1` contains 13 exact duplicate rows (the same household appearing twice with identical values). These are dropped, since counting the same household twice would double its influence on training and bias the churn rate slightly.

In [2]:
before = len(t1)
t1 = t1.drop_duplicates().copy()
print(f"table1: {before} rows -> {len(t1)} rows ({before - len(t1)} exact duplicates removed)")

table1: 4485 rows -> 4472 rows (13 exact duplicates removed)


### 2.2 Inconsistent categorical text

Every categorical column across all six tables has the same real-world messiness: the same category appears in several forms, for example `plan_tier` contains `'Standard'`, `'STANDARD'`, `'standard'`, and `' Standard '` as four different strings for one category. Left as-is, a model (and especially one-hot encoding) would treat these as different categories entirely. A single helper strips whitespace, collapses internal double-spaces, lowercases, and capitalises the first letter, giving one consistent spelling per category, applied to every categorical column that needs it.

In [3]:
def clean_cat(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\s+", " ", regex=True)
             .str.lower().str.capitalize())

for col in ["plan_tier", "payment_method", "payment_status"]:
    before_n = t1[col].nunique()
    t1[col] = clean_cat(t1[col])
    print(f"table1.{col}: {before_n} raw spellings -> {t1[col].nunique()} categories")

t1["upgrade_downgrade_flag"] = clean_cat(t1["upgrade_downgrade_flag"]).fillna("None")

for col in ["event_type", "device_type"]:
    t2[col] = clean_cat(t2[col])
for col in ["contact_channel", "reason_code", "resolution_status"]:
    t4[col] = clean_cat(t4[col])
t5["genre"] = clean_cat(t5["genre"])

table1.plan_tier: 12 raw spellings -> 3 categories
table1.payment_method: 15 raw spellings -> 3 categories
table1.payment_status: 11 raw spellings -> 3 categories


### 2.3 Mixed date formats in the viewing log

`table2.event_timestamp` mixes several formats: ISO (`2026-08-24 06:21:18`), slash-separated with the year first (`2025/11/26`), and slash-separated with the year last in both day-first and month-first order (`29/05/2026`, `03/22/2025 16:58`). A single `pd.to_datetime` call cannot handle this reliably, since pandas infers one format for the whole column and silently mis-parses whatever doesn't match (an early version of this cell parsed over 2,000 timestamps into the future this way, which is how the bug below was caught).

The fix tries the most specific, unambiguous formats first (ISO and year-first), then falls back to day-first for whatever's left, since South Africa (STADIOchoice's largest market, per the SS1 briefing pack) uses day-first dates by convention, and finally month-first for any remaining genuinely ambiguous cases.

In [4]:
def parse_mixed_datetime(s):
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d %H:%M", "%Y/%m/%d"]:
        remaining = out.isna() & s.notna()
        if not remaining.any():
            break
        out.loc[remaining] = pd.to_datetime(s[remaining], format=fmt, errors="coerce")
    remaining = out.isna() & s.notna()
    out.loc[remaining] = pd.to_datetime(s[remaining], format="mixed", dayfirst=True, errors="coerce")
    remaining = out.isna() & s.notna()
    out.loc[remaining] = pd.to_datetime(s[remaining], format="mixed", dayfirst=False, errors="coerce")
    return out

t2["event_timestamp"] = parse_mixed_datetime(t2["event_timestamp"])
assert t2["event_timestamp"].isna().sum() == 0, "some timestamps still failed to parse"

### 2.4 Implausible values: events after the extract date

The client's README warns to expect "occasional out-of-range or implausible values from data-capture errors". Checking for exactly that: a handful of viewing events are timestamped *after* the 2026-08-31 extract date, which is impossible, since no event can be logged after the data was pulled. These are dropped rather than silently left in, since they would otherwise quietly distort any recency-based feature.

In [5]:
impossible = t2["event_timestamp"] > EXTRACT_DATE + pd.Timedelta(days=1)
print(f"Dropping {impossible.sum()} events timestamped after the extract date")
t2 = t2[~impossible].copy()

Dropping 5 events timestamped after the extract date


### 2.5 Avoiding leakage

The client's README is explicit about this, and it matters: `churn_date`, `tenure_months` (measured up to the churn date for churned households, so not even comparable across the two classes), and the whole of `table3_cancellation_retention.csv` only exist **after** a household has already decided to leave. Using any of them as predictors would not build a churn predictor at all, it would just teach the model to recognise its own label. `table3` is read for context (the briefing pack's note that retention-accepted households show up there while still active in `table1` is a genuinely useful counterfactual to know about), but no column from it enters the feature set. In place of `tenure_months`, `subscription_age_months` is computed from `subscription_start_date` relative to the fixed extract date, which is available identically for active and churned households alike.

In [6]:
t1["subscription_start_date"] = pd.to_datetime(t1["subscription_start_date"])
t1["subscription_age_months"] = ((EXTRACT_DATE - t1["subscription_start_date"]).dt.days / 30.44).round(1)

LEAKAGE_COLUMNS = ["churn_date", "tenure_months", "subscription_start_date", "billing_cycle_date"]
print("Columns excluded from the feature set as post-outcome or non-predictive:", LEAKAGE_COLUMNS)

Columns excluded from the feature set as post-outcome or non-predictive: ['churn_date', 'tenure_months', 'subscription_start_date', 'billing_cycle_date']


### 2.6 Feature engineering from the behavioural tables

SS2's Part D recommendations flagged that the Telco proxy dataset could only offer static service flags, and that STADIOchoice's real viewing, search, and support data would let a household-level model use genuine behavioural signal instead. This is where that happens: `table2` (joined to `table5` for genre and sports-content flags), `table4`, and `table6` are each aggregated to one row per household and joined onto `table1`.

- From viewing logs: total events, plays, abandons, distinct devices, days since the last event (recency), total watch time, distinct titles watched, genre diversity, and the fraction of watched content that is sport, since the SS1 briefing pack specifically named sports-tournament endings as a known churn trigger.
- From support contacts: contact count, and counts of escalated/unresolved outcomes, a household with repeated unresolved issues is a plausible churn risk.
- From search queries: search count and the zero-result rate, which the client's README calls out directly as "a direct measure of unmet content demand and a documented churn precursor".

Households with no rows in a given table (never searched, never contacted support) get 0 for those features rather than a missing value, since no activity is a real, meaningful state here, not an unknown one.

In [7]:
content_events = t2[t2["event_type"].isin(["Play", "Pause"])]
t2_joined = t2.merge(t5[["title_id", "genre", "is_sports_content"]], on="title_id", how="left")
content_joined = t2_joined[t2_joined["event_type"].isin(["Play", "Pause"])]

viewing_agg = t2.groupby("household_id").agg(
    total_events=("event_type", "count"),
    play_count=("event_type", lambda s: (s == "Play").sum()),
    abandon_count=("event_type", lambda s: (s == "Abandon").sum()),
    distinct_devices=("device_id", "nunique"),
    last_event_date=("event_timestamp", "max"),
)
viewing_agg["days_since_last_event"] = (EXTRACT_DATE - viewing_agg["last_event_date"]).dt.days
viewing_agg = viewing_agg.drop(columns=["last_event_date"])

content_agg = content_joined.groupby("household_id").agg(
    total_watch_seconds=("watch_duration_seconds", "sum"),
    distinct_titles=("title_id", "nunique"),
    genre_diversity=("genre", "nunique"),
    content_event_count=("is_sports_content", "count"),
    sports_event_count=("is_sports_content", lambda s: (s == True).sum()),
)
content_agg["sports_fraction"] = (content_agg["sports_event_count"] / content_agg["content_event_count"]).fillna(0)
content_agg = content_agg.drop(columns=["content_event_count", "sports_event_count"])

support_agg = t4.groupby("household_id").agg(
    support_contacts=("reason_code", "count"),
    escalated_count=("resolution_status", lambda s: (s == "Escalated").sum()),
    unresolved_count=("resolution_status", lambda s: (s == "Unresolved").sum()),
)

search_agg = t6.groupby("household_id").agg(
    search_count=("result_count", "count"),
    zero_result_count=("result_count", lambda s: (s == 0).sum()),
)
search_agg["zero_result_rate"] = search_agg["zero_result_count"] / search_agg["search_count"]

print("Engineered feature groups ready:", viewing_agg.shape, content_agg.shape, support_agg.shape, search_agg.shape)

Engineered feature groups ready: (4472, 5) (4184, 4) (1890, 3) (3896, 3)


### 2.7 Assembling the feature matrix

Joining the engineered features onto the cleaned `table1`, one-hot encoding the remaining categorical columns, and defining the target as `needs_review`-style binary flag: 1 means churned (`is_active == False`), matching the client's own definition.

In [8]:
features = t1.drop(columns=LEAKAGE_COLUMNS).set_index("household_id")
for agg in [viewing_agg, content_agg, support_agg, search_agg]:
    features = features.join(agg, how="left")

fill_zero_cols = [
    "total_events", "play_count", "abandon_count", "distinct_devices", "days_since_last_event",
    "total_watch_seconds", "distinct_titles", "genre_diversity", "sports_fraction",
    "support_contacts", "escalated_count", "unresolved_count",
    "search_count", "zero_result_count", "zero_result_rate",
]
features[fill_zero_cols] = features[fill_zero_cols].fillna(0)
# Households with no viewing events at all have no recency value; treat them as having been
# inactive since they joined, i.e. their full subscription age in days.
no_events = features["days_since_last_event"].isna()
features.loc[no_events, "days_since_last_event"] = features.loc[no_events, "subscription_age_months"] * 30.44

y = (~features["is_active"]).astype(int)
X = features.drop(columns=["is_active"])
X = pd.get_dummies(X, columns=["plan_tier", "payment_method", "payment_status", "upgrade_downgrade_flag"], drop_first=True)

print(f"Final feature matrix: {X.shape[0]} households, {X.shape[1]} features")
print(f"Churn rate in the cleaned data: {y.mean():.1%}")
assert not set(LEAKAGE_COLUMNS) & set(X.columns), "a leakage column ended up in the feature set"

Final feature matrix: 4472 households, 26 features
Churn rate in the cleaned data: 18.9%


### 2.8 Train, validation, and test split

A three-way split is used here, unlike SS2's simple train/test split, since the validation set lets model training use early stopping against genuinely unseen data (see Section 3) rather than just a fixed number of boosting rounds. The test set is held out completely and only touched in Section 4. All three splits are stratified on the target so each keeps roughly the same ~18.9% churn rate as the full dataset.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

for name, ys in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: {len(ys)} households, churn rate {ys.mean():.1%}")

train: 3130 households, churn rate 18.9%
validation: 671 households, churn rate 18.9%
test: 671 households, churn rate 18.9%


## 3. Model training

SS2 compared Logistic Regression against XGBoost on the Telco proxy dataset, and XGBoost won on every metric with the difference confirmed statistically significant (exact McNemar's test, p=0.0108, see SS2's Comparison.MD). That is the "final method" this section applies here.

**Hyperparameters are kept the same as SS2** (`n_estimators=200`, `max_depth=4`, `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.8`) for direct alignment between the two submissions, this is the same architecture, not a different model wearing the same name. Two things do change, deliberately and for good reason:

- **`scale_pos_weight` is recalculated from this data's actual class balance** (about 4.3:1 non-churned to churned here, versus about 2.77:1 for the Telco proxy task). Reusing SS2's value would under-correct for the real imbalance, since STADIOchoice's churn rate (18.9%) is lower than Telco's (26.5%).
- **Early stopping against the validation set is added**, something SS2 could not do with only a train/test split. Training stops once validation log-loss has not improved for 20 rounds, which guards against overfitting the 3,130-row training set without ever touching the test set to make that decision.

In [10]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (recalculated for this data): {scale_pos_weight:.3f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    early_stopping_rounds=20,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(f"Best iteration: {model.best_iteration} (of a maximum 200)")
print(f"Best validation log-loss: {model.best_score:.4f}")

scale_pos_weight (recalculated for this data): 4.278


Best iteration: 198 (of a maximum 200)
Best validation log-loss: 0.2777
